### The goal of this notebook is exploratory data test for a better understanding if the data. 
### There is not a project purpose.

---

In [1]:
import pandas as pd
import yaml
import openpyxl

In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [3]:
# P_A1_25_clean.xlsx is not in the repo, so rebuild it from the tested_and_ready CSV (same 7524 rows)
# using the column names the rest of this notebook expects.
P_A1_25_clean = (
    pd.read_csv(cfg["P_A1_25_ready_csv_path"], encoding="utf-8-sig")
    .drop(columns="plant")
    .rename(columns={
        "cement_4av": "cement_iv_av",
        "ad_complast_mr": "aditive_mr",
        "ad_dynamon_sx": "aditive_sx3",
        "ad_auramix": "aditive_auramix",
    })
)

In [5]:
# How many zeros each column has and what percentage of all rows that represents.
zero_counts = (P_A1_25_clean == 0.0).sum()
zero_pct = zero_counts / len(P_A1_25_clean) * 100

pd.DataFrame({
    "zero_count": zero_counts,
    "zero_pct": zero_pct.round(2)
})

# Add the ones with values and total knowing that must be the total of rows always, 7524.

,zero_count,zero_pct
date,0,0.00
product,0,0.00
volume,2,0.03
cement_42_5r,54,0.72
cement_iv_av,7510,99.81
arid_04,37,0.49
arid_8_14,280,3.72
arid_12_20,1959,26.04
water,162,2.15
aditive_mr,6773,90.02


In [6]:
# Display the rows where the volume column value is 0.
P_A1_25_clean[P_A1_25_clean["volume"] == 0.0]

,date,product,volume,cement_42_5r,cement_iv_av,arid_04,arid_8_14,arid_12_20,water,aditive_mr,aditive_sx3,aditive_auramix
57,2025-12-23,HA-25/B/20/XC2/1,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
4379,2025-06-10,HA-25/B/20/XC2/1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.9


In [7]:
# These two rows very likely belong to a wrong way to add water and additive respectivley o the reason it could be a simply working test. So we procceed to drop them.
P_A1_25_clean.drop(P_A1_25_clean[P_A1_25_clean["volume"] == 0.0].index, inplace=True)

In [8]:
print(P_A1_25_clean.shape)

(7522, 12)


In [9]:
P_A1_25_clean["product"].unique()

<ArrowStringArray>
[               'HA-25/B/20/IIa/1',                'HA-25/B/20/XC1/1',
                  'HM-20/B/20/I/1',                'HA-30/F/14/XC1/1',
     'IMPRESO METRO  HM-20/F/14/I',          'IMPRESO.HA-25/B/14/IIa',
                  'SECOMD-200/S/4',                'HA-35/F/14/XC1/1',
         'PILARES  HA-25/B/20/IIa',                'HA-25/B/20/XC2/1',
                'HA-25/B/14/IIa/1',                'HA-25/B/14/XC2/1',
                   'HL-150/B/20/1',                'HA-30/F/14/XC2/1',
                 'HA-25/B/20/X0/1',                'HA-25/F/20/XC2/1',
          'IMPRESO.HA-25/F/14/IIa',                'HA-25/F/14/IIa/1',
                'HA-30/B/20/IIa/1',                'HA-30/B/20/XC3/1',
     'PILARES MURO HA-25/B/20/XC1',                  'RES.H-150/B/15',
                  'RES.D-100/B/20',     'CARRIL BICI HF-3,5/F/20/IIa',
 'PILARES    MURO  HA-25/B/20/XC2',                'HA-30/B/14/XC2/1',
          'BOMBEABLE  HL-150/B/20',            'IMPRESO.HM

### CEMENT Outlier Detection. Median and count and identify one 20% higher over or below the median. (IQR method discarded)

In [10]:
# Build P_A1_25_cement_consumption_anomaly from a copy — P_A1_25_clean is not modified.
P_A1_25_cement_consumption_anomaly = P_A1_25_clean.copy()

P_A1_25_cement_consumption_anomaly["cement_per_metre"] = (
    (P_A1_25_cement_consumption_anomaly["cement_42_5r"] + P_A1_25_cement_consumption_anomaly["cement_iv_av"]) / P_A1_25_cement_consumption_anomaly["volume"]
).round(2)

P_A1_25_cement_consumption_anomaly["median_product_cons"] = (
    P_A1_25_cement_consumption_anomaly.groupby("product")["cement_per_metre"].transform("median")
).round(2)

P_A1_25_cement_consumption_anomaly["pct_vs_median"] = (
    ((P_A1_25_cement_consumption_anomaly["cement_per_metre"] - P_A1_25_cement_consumption_anomaly["median_product_cons"]) / P_A1_25_cement_consumption_anomaly["median_product_cons"]) * 100
).round(1)

P_A1_25_cement_consumption_anomaly = P_A1_25_cement_consumption_anomaly[
    (P_A1_25_cement_consumption_anomaly["pct_vs_median"] >= 10) | (P_A1_25_cement_consumption_anomaly["pct_vs_median"] <= -10)
][["date", "product", "volume", "cement_per_metre", "pct_vs_median", "median_product_cons",
   "cement_42_5r", "cement_iv_av", "arid_04", "arid_8_14", "arid_12_20",
   "water", "aditive_mr", "aditive_sx3", "aditive_auramix"]].sort_values(
    ["product", "pct_vs_median"], ascending=[True, False]
).reset_index(drop=True)

print(f"Rows with |pct_vs_median| >= 10%: {len(P_A1_25_cement_consumption_anomaly)}")
P_A1_25_cement_consumption_anomaly


Rows with |pct_vs_median| >= 10%: 144


,date,product,volume,cement_per_metre,pct_vs_median,median_product_cons,cement_42_5r,cement_iv_av,arid_04,arid_8_14,arid_12_20,water,aditive_mr,aditive_sx3,aditive_auramix
0,2025-11-26,BOMBEABLE HL-150/B/20,6.0,200.33,33.6,150.00,1202.0,0.0,8280.0,1427.0,3043.0,702.0,0.0,0.0,7.2
1,2025-10-01,BOMBEABLE HL-150/B/20,2.0,106.00,-29.3,150.00,212.0,0.0,1724.0,0.0,3034.0,239.0,0.0,0.0,0.0
2,2025-09-11,HA-25/B/14/IIa/1,1.5,750.00,170.9,276.86,1125.0,0.0,2005.0,1275.0,0.0,202.0,0.0,1.4,2.9
3,2025-10-09,HA-25/B/14/IIa/1,4.0,314.75,13.7,276.86,1259.0,0.0,4735.0,3338.0,0.0,569.0,0.0,3.9,8.7
4,2025-10-09,HA-25/B/14/IIa/1,5.5,306.36,10.7,276.86,1685.0,0.0,6664.0,4620.0,0.0,786.0,0.0,5.3,12.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,2025-11-10,RES.D-100/B/20,2.5,120.00,20.1,99.90,300.0,0.0,3123.0,1423.0,838.0,320.0,0.0,0.0,1.9
140,2025-07-17,RES.D-100/B/20,5.0,84.40,-15.5,99.90,422.0,0.0,5946.0,1689.0,2996.0,709.0,0.0,0.0,2.4
141,2025-05-12,RES.D-100/B/20,1.0,81.00,-18.9,99.90,81.0,0.0,1174.0,375.0,796.0,136.0,0.0,0.0,0.2
142,2025-11-19,RES.D-100/B/20,4.0,0.00,-100.0,99.90,0.0,0.0,4872.0,1866.0,1880.0,845.0,0.0,0.0,0.0


In [13]:
out_path = ROOT / "data" / "processed" / "P_A1_25_cement_consumption_anomaly.xlsx"
out_path.parent.mkdir(parents=True, exist_ok=True)
P_A1_25_cement_consumption_anomaly.to_excel(out_path, index=False, engine="openpyxl")


In [15]:
# Which products have anomalies?
P_A1_25_cement_consumption_anomaly["product"].unique()

<ArrowStringArray>
[         'BOMBEABLE  HL-150/B/20',                'HA-25/B/14/IIa/1',
                  'HA-25/B/20/I/1',                'HA-25/B/20/IIa/1',
                 'HA-25/B/20/X0/1',                'HA-25/B/20/XC1/1',
                'HA-25/B/20/XC2/1',                'HA-25/B/20/XC3/1',
                'HA-25/F/14/IIa/1',                'HA-25/F/14/XC2/1',
                 'HA-25/F/20/X0/1',                'HA-25/F/20/XC1/1',
                'HA-30/B/20/XA1/1',                'HA-30/B/20/XC1/1',
                'HA-30/B/20/XC2/1',                'HA-30/B/20/XC3/1',
                'HA-30/F/14/XC2/1',                'HA-35/F/14/XC1/1',
                   'HL-150/B/20/1',                  'HM-20/B/14/I/1',
                  'HM-20/B/20/I/1',     'IMPRESO METRO  HM-20/F/14/I',
          'IMPRESO.HA-25/B/20/IIa', 'PILARES    MURO  HA-25/B/20/XC2',
                  'RES.D-100/B/20']
Length: 25, dtype: str

In [16]:
# Frequency of anomalies (over 10% consumption median deviation) by product.
P_A1_25_cement_consumption_anomaly.groupby("product").size().reset_index(name="anomaly_count").sort_values("anomaly_count", ascending=False)


,product,anomaly_count
20,HM-20/B/20/I/1,44
3,HA-25/B/20/IIa/1,23
6,HA-25/B/20/XC2/1,15
18,HL-150/B/20/1,12
5,HA-25/B/20/XC1/1,12
24,RES.D-100/B/20,5
14,HA-30/B/20/XC2/1,4
19,HM-20/B/14/I/1,4
8,HA-25/F/14/IIa/1,3
1,HA-25/B/14/IIa/1,3


In [17]:
# Frequency of anomalies (over 10% consumption median deviation) by product.
P_A1_25_cement_consumption_anomaly.groupby("product").size().reset_index(name="anomaly_count").sort_values("anomaly_count", ascending=False)


,product,anomaly_count
20,HM-20/B/20/I/1,44
3,HA-25/B/20/IIa/1,23
6,HA-25/B/20/XC2/1,15
18,HL-150/B/20/1,12
5,HA-25/B/20/XC1/1,12
24,RES.D-100/B/20,5
14,HA-30/B/20/XC2/1,4
19,HM-20/B/14/I/1,4
8,HA-25/F/14/IIa/1,3
1,HA-25/B/14/IIa/1,3
